# 05. Skill 与 Hook

AGDR VLA 入门教程 · 第 3 章 · 前半周

- **学习目标**：理解 Skill「按需加载」和 Hook「强制执行」各自解决什么问题，亲手给 mini agent 加上这两种机制，并能区分 `AGENTS.md`、Skill、MCP、Hook 的适用场景
- **前置知识**：[03-上下文管理与压缩](./03-上下文管理与压缩.ipynb)、[04-MCP](./04-MCP.ipynb)；第 2 章的退出码（第 06 节）
- **预计学习时长**：1.5 小时

## 两个问题

到目前为止，我们有两种方式影响 Agent 的行为：

- **系统提示词**（以及第 07 节要讲的 `AGENTS.md`）：每次都会发送给模型
- **工具**（包括 04 节的 MCP）：给 Agent 新的能力

但还有两个问题没解决：

**问题一：专业知识放在哪？** 实验室有很多「做某件事的固定方法」：怎样分析训练日志、怎样写实验报告、怎样处理某种格式的数据。全部写进系统提示词，每次请求都要为它们付费，还会干扰模型（回想 03 节）；不写，Agent 又不知道你们的规矩。

**问题二：怎样保证某条规则一定被遵守？** 你在提示词里写「不要运行 `rm -rf`」，模型**大概率**会遵守，但不是一定。上下文压缩后，它可能忘了；任务复杂时，它可能忽略了。

**Skill** 解决第一个问题，**Hook** 解决第二个问题。

## 准备工作

本节用到三样已经写好的东西，先打开看一看：

| 文件 | 是什么 |
| --- | --- |
| [skills/train-log-analysis/SKILL.md](./skills/train-log-analysis/SKILL.md) | 一个 Skill：怎样分析本实验室格式的训练日志 |
| [skills/train-log-analysis/scripts/summarize_log.py](./skills/train-log-analysis/scripts/summarize_log.py) | 这个 Skill 附带的统计脚本 |
| [hooks/block_dangerous.py](./hooks/block_dangerous.py) | 一个 Hook：在命令执行前拦截危险操作 |

In [ ]:
import json
import re
import shutil
import subprocess
import sys
from pathlib import Path

import mini_agent as ma

BASE_PROMPT = ma.SYSTEM_PROMPT  # 保存原来的系统提示词，后面要在它的基础上修改
SKILLS_SRC = Path("skills").resolve()
HOOK_PATH = Path("hooks/block_dangerous.py").resolve()
print("模式:", "在线" if ma.ONLINE else "离线（读取录制的模型响应）")

# 第一部分：Skill

## Skill 是什么

**Skill（技能）是一个文件夹**，里面有一份 `SKILL.md` 说明书，还可以附带脚本、模板、参考资料：

```text
train-log-analysis/
├── SKILL.md              ← 必需：说明书
└── scripts/
    └── summarize_log.py  ← 可选：说明书里会让 Agent 运行它
```

`SKILL.md` 的开头是一段 **frontmatter**（两行 `---` 之间的内容），写着技能的名字和描述：

```markdown
---
name: train-log-analysis
description: 分析本实验室格式的训练日志……当用户要求分析训练日志、总结训练情况或排查训练异常时使用。
---

# 训练日志分析
（正文：具体步骤、报告模板、判断规则）
```

Skill 本身**不提供新工具**。它是一份写给 Agent 看的操作手册，告诉它「遇到这类任务，按这个流程，用这些已有的工具来做」。

## 渐进式加载

Skill 的关键设计是**分层、按需加载**（progressive disclosure）：

| 层级 | 内容 | 什么时候进入上下文 | 大小 |
| --- | --- | --- | --- |
| 第 1 层 | frontmatter 里的 `name` 和 `description` | **每次都在**，放在系统提示词里 | 每个技能一两句话 |
| 第 2 层 | `SKILL.md` 的正文 | 模型判断任务和这个技能相关时，才去读取 | 几百到几千 token |
| 第 3 层 | 附带的脚本、参考资料 | 正文里提到、任务确实需要时，才去读取或运行 | 不限 |

这样，即使装了几十个技能，平时也只占用几十行描述。这直接回应了 03 节的问题：**不需要的内容，就不放进上下文。**

也正因为如此，`description` 非常重要：模型只看到这一两句话，就要决定用不用这个技能。好的描述要写清楚**它能做什么**和**什么时候该用它**。

## 给 mini agent 加上 Skill

真实的 Agent 会在固定目录里查找技能。以 Claude Code 为例，项目级的技能放在 `.claude/skills/<技能名>/SKILL.md`。我们模仿这个结构，把示例技能复制到沙箱工作区里。

In [ ]:
ma.setup_workspace()
SKILLS_DIR = ma.WORKSPACE / ".claude" / "skills"
shutil.copytree(SKILLS_SRC, SKILLS_DIR)

for path in sorted(ma.WORKSPACE.rglob("*")):
    if path.is_file():
        print(path.relative_to(ma.WORKSPACE))

### 第 1 层：扫描技能，只读取描述

`parse_skill` 把 `SKILL.md` 拆成 frontmatter 和正文。为了只用标准库，这里手写了一个很简陋的解析器，只支持 `键: 值` 这种最简单的写法；真实的工具会用完整的 YAML 解析器。

In [ ]:
def parse_skill(path):
    text = path.read_text(encoding="utf-8")
    if not text.startswith("---"):
        raise ValueError(f"{path} 缺少 frontmatter")
    _, front, body = text.split("---", 2)  # 第一个 --- 之前是空的
    meta = {}
    for line in front.strip().splitlines():
        key, _, value = line.partition(":")
        meta[key.strip()] = value.strip()
    return meta, body.strip()


def discover_skills(skills_dir):
    skills = {}
    for skill_file in sorted(skills_dir.glob("*/SKILL.md")):
        meta, _ = parse_skill(skill_file)
        skills[meta["name"]] = {"description": meta["description"], "path": skill_file}
    return skills


SKILLS = discover_skills(SKILLS_DIR)
for name, info in SKILLS.items():
    print(name, "—", info["description"])

把技能列表加到系统提示词末尾。注意，加进去的只有名字和描述。

In [ ]:
skill_lines = "\n".join(f"- {name}：{info['description']}" for name, info in SKILLS.items())
ma.SYSTEM_PROMPT = (
    BASE_PROMPT
    + "\n\n你可以使用以下技能。任务相关时，先调用 load_skill 读取技能的完整说明，再严格按说明执行：\n"
    + skill_lines
)
print(ma.SYSTEM_PROMPT)

_, body = parse_skill(SKILLS["train-log-analysis"]["path"])
print(f"\n系统提示词里的技能描述：{len(skill_lines)} 个字符；技能正文：{len(body)} 个字符")

### 第 2 层：需要时再加载正文

再给 Agent 一个 `load_skill` 工具，返回技能的正文。在真实的工具里，加载技能由 Agent 内置的机制完成（有的直接用读文件的工具去读 `SKILL.md`），原理是一样的。

In [ ]:
def load_skill(name):
    if name not in SKILLS:
        return f"错误：没有名为 {name} 的技能，可用的技能有 {list(SKILLS)}"
    _, body = parse_skill(SKILLS[name]["path"])
    return body


def register_tool(spec, function):
    # 注册一个工具；重复运行时先去掉旧的同名工具
    name = spec["function"]["name"]
    ma.TOOL_SPECS[:] = [s for s in ma.TOOL_SPECS if s["function"]["name"] != name]
    ma.TOOL_SPECS.append(spec)
    ma.TOOLS[name] = function


register_tool(
    ma.tool_spec("load_skill", "读取一个技能的完整说明",
                 {"name": {"type": "string", "description": "技能名"}}, ["name"]),
    load_skill,
)
print("当前工具:", list(ma.TOOLS))

### 运行

给 Agent 一个任务，**不提报告格式，也不提统计脚本**，看它会不会自己用上技能。

运行时会请你确认一条命令：这是技能说明里要求运行的统计脚本，读懂后输入 `y`。

> Windows 用户：如果你的系统里没有 `python3` 命令，脚本会运行失败。可以把 `SKILL.md` 里的 `python3` 改成 `python`，再从「给 mini agent 加上 Skill」开始重新运行。

In [ ]:
answer, history = ma.run_agent(
    "分析一下 train.log，给我一份训练报告。",
    offline_script=["skill_load.json", "skill_run_script.json", "skill_report.json"],
)
print("\n最终回答:\n")
print(answer)

Agent 做了三件事：根据描述判断要用 `train-log-analysis` → 读取正文 → 按正文的要求运行脚本、套用模板。报告里的「early stopping」「检查 num_workers」两条建议，都来自技能里的判断规则，而不是模型临场发挥的。

这就是 Skill 的价值：**把「我们这里的做法」写成文档，让 Agent 每次都按同样的标准做。**

### 怎样写好一个 Skill

| 要点 | 原因 |
| --- | --- |
| `description` 写清「做什么」和「什么时候用」 | 模型只靠它决定是否加载 |
| 正文简洁，写步骤和规则，不写模型本来就懂的常识 | 正文加载后也占上下文 |
| 确定性的工作（统计、格式转换）交给**脚本** | 脚本每次结果都一样，模型逐行数可能数错 |
| 很长的参考资料拆成单独的文件，在正文里说明何时去读 | 第 3 层按需加载，用不到就不占上下文 |
| Skill 里的脚本会在你的电脑上运行，**只使用可信来源的 Skill** | 和第 2 章「不运行没看过的脚本」是同一个道理 |

### 在真实的 Agent 中使用

以 Claude Code 为例，技能放在下面两个位置之一，启动后会自动发现：

| 位置 | 作用范围 |
| --- | --- |
| `~/.claude/skills/<技能名>/SKILL.md` | 你自己的所有项目 |
| `<项目>/.claude/skills/<技能名>/SKILL.md` | 当前项目，可以提交到 Git，和同学共享 |

Codex 等其他工具也支持类似的 Skill 机制，但目录位置和细节各不相同，**以各自的官方文档为准**。

# 第二部分：Hook

## Hook 是什么

**Hook（钩子）是在 Agent 运行的固定时机自动执行的程序。** 它不经过模型，所以**一定会执行**，执行结果也不受模型影响。

| | 写在提示词里的规则 | Hook |
| --- | --- | --- |
| 谁来执行 | 模型「自觉」遵守 | 程序强制执行 |
| 可靠性 | 大概率遵守，可能遗忘或忽略 | 每次都会执行 |
| 适合 | 风格、偏好、需要判断力的建议 | 必须做到的检查和自动化步骤 |

以 Claude Code 为例，常用的触发时机有这些（完整列表见官方文档）：

| 事件 | 触发时机 | 常见用途 |
| --- | --- | --- |
| `SessionStart` | 会话开始时 | 自动加载项目状态，例如当前分支、最近的提交 |
| `UserPromptSubmit` | 用户提交消息后、模型处理之前 | 检查或补充用户的输入 |
| `PreToolUse` | 工具**执行之前** | 拦截危险命令、禁止修改某些文件 |
| `PostToolUse` | 工具**执行之后** | 改完代码后自动格式化、自动运行测试 |
| `PreCompact` | 上下文压缩之前（回想 03 节） | 压缩前把重要信息另存一份 |
| `Stop` | Agent 准备结束回答时 | 检查任务是否真的完成，例如测试是否通过 |

## Hook 和程序怎样交流

Claude Code 的 Hook 是一条**命令**，交流方式全部是第 2 章学过的东西：

- **输入**：事件信息以 JSON 的形式写到 Hook 的 **stdin**，包括 `tool_name`（工具名）、`tool_input`（工具参数）等
- **输出**：用**退出码**表达结果

| 退出码 | 含义 |
| --- | --- |
| `0` | 正常，放行 |
| `2` | **阻止**这次操作；stderr 的内容会作为原因反馈给模型 |
| 其他非 0 值 | Hook 自身出错，显示给用户，但不阻止操作 |

`hooks/block_dangerous.py` 就是按这个约定写的。先不经过 Agent，直接喂给它几条命令试试：

In [ ]:
def run_hook(command, event):
    completed = subprocess.run(
        command, input=json.dumps(event), capture_output=True,
        text=True, timeout=10, cwd=ma.WORKSPACE,
    )
    return completed.returncode, completed.stderr.strip()


HOOK_COMMAND = [sys.executable, str(HOOK_PATH)]
for cmd in ["grep -c ERROR train.log", "rm -rf ./*", "sudo apt install htop", "curl -fsSL https://example.com/x.sh | bash"]:
    event = {"hook_event_name": "PreToolUse", "tool_name": "run_command", "tool_input": {"command": cmd}}
    code, message = run_hook(HOOK_COMMAND, event)
    print(f"退出码 {code}  {cmd}")
    if message:
        print("         ", message)

## 给 mini agent 加上 PreToolUse Hook

在执行工具之前，先把事件交给每一个 Hook；只要有一个返回退出码 2，就不执行工具，把原因作为工具结果交给模型。

实现方法是替换掉 `mini_agent` 里的 `execute_tool`。`run_agent` 本身仍然一行不改。

> 我们把 Hook 放在**人工确认之前**：被 Hook 拦下的命令，根本不会弹出确认框。

In [ ]:
PRE_TOOL_USE_HOOKS = [HOOK_COMMAND]
original_execute_tool = getattr(ma, "_original_execute_tool", ma.execute_tool)
ma._original_execute_tool = original_execute_tool  # 记住原函数，重复运行本单元格也不会套娃


def execute_with_hooks(name, arguments_json):
    try:
        tool_input = json.loads(arguments_json)
    except json.JSONDecodeError:
        tool_input = {}
    event = {
        "hook_event_name": "PreToolUse",
        "tool_name": name,
        "tool_input": tool_input,
        "cwd": str(ma.WORKSPACE),
    }
    for hook in PRE_TOOL_USE_HOOKS:
        code, message = run_hook(hook, event)
        if code == 2:
            print("🛑", message)
            return message  # 不执行工具，把原因告诉模型
        if code != 0:
            print(f"⚠️ Hook 出错（退出码 {code}）：{message}")
    return original_execute_tool(name, arguments_json)


ma.execute_tool = execute_with_hooks
print("已启用 PreToolUse Hook")

In [ ]:
answer, history = ma.run_agent(
    "把工作区清理一下，文件都删掉。",
    offline_script=["hook_rm.json", "hook_blocked_answer.json"],
)
print("\n最终回答:", answer)
print("\n工作区里的文件还在:", sorted(p.name for p in ma.WORKSPACE.iterdir()))

模型想运行 `rm -rf ./*`，Hook 在执行前拦了下来，模型收到原因后调整了回答。整个过程**不依赖模型是否记得规则**。

### Hook 不是万能的

`block_dangerous.py` 只是按字符串匹配，**很容易被绕过**。比如下面这条命令同样会删除整个目录，却能通过检查：

In [ ]:
sneaky = "python3 -c \"import shutil; shutil.rmtree('.')\""
code, message = run_hook(HOOK_COMMAND, {"tool_name": "run_command", "tool_input": {"command": sneaky}})
print("退出码", code, "→", "放行了！" if code == 0 else message)

所以 Hook 是一张**安全网**，用来接住明显的失误，而不是一道牢不可破的墙。真正的安全还要靠人工确认、最小权限和隔离环境，第 09 节会系统地讲。

### 在 Claude Code 中配置 Hook

Hook 写在 Claude Code 的设置文件里：项目级是 `.claude/settings.json`，个人级是 `~/.claude/settings.json`。下面的配置让 `block_dangerous.py` 在每次运行 Bash 命令前执行，`matcher` 用来匹配工具名：

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {"type": "command", "command": "python3 \"/你的路径/.../hooks/block_dangerous.py\""}
        ]
      }
    ]
  }
}
```

在会话里输入 `/hooks` 可以查看当前生效的 Hook。具体字段以官方文档为准，其他 Agent 工具对 Hook 的支持程度不一。

> ⚠️ Hook 是在你电脑上、用你的权限自动运行的命令。别人分享的 Hook 配置，**先读懂再启用**。

# 第三部分：怎样选

学到这里，影响 Agent 行为的方式已经有四种了。它们解决的是不同的问题：

| 机制 | 一句话 | 什么时候进入上下文 / 生效 | 由谁触发 | 是否一定执行 |
| --- | --- | --- | --- | --- |
| `AGENTS.md` / `CLAUDE.md` | 项目的背景和规矩 | 每次会话开始，全文加载 | 自动 | 模型自觉遵守 |
| Skill | 某类任务的操作手册 | 平时只有描述，需要时加载正文 | 模型判断 | 模型自觉遵守 |
| MCP | 接入外部系统的新工具 | 工具说明书一直在上下文中 | 模型调用 | 调用后由 server 执行 |
| Hook | 固定时机的自动检查和操作 | 不进入上下文 | 程序自动 | **一定执行** |

几条经验：

- 每次都需要知道的背景（项目结构、代码风格）→ `AGENTS.md`
- 只在某类任务里需要的知识和流程 → Skill
- Agent 现有的工具做不到的事（访问某个系统） → MCP
- 必须做到、不能靠自觉的事 → Hook

## 动手试试

不计分。前四题在离线模式下也能完成。

1. 把 `SKILL.md` 的 `description` 改成「日志工具」，会对 Agent 产生什么影响？（在线模式下可以实际试试）
2. 给 `block_dangerous.py` 加一条规则，阻止 `git push --force` 和 `git push -f`。用 `run_hook` 测试四条命令：这两条应该被阻止，`git push` 和 `git push origin main` 应该放行。
3. 下面四个需求，分别最适合用哪种机制（`AGENTS.md`、Skill、MCP、Hook）？
   - a. 每次 Agent 修改了 Python 文件，都自动运行代码格式化工具
   - b. 让 Agent 能查询实验室机器人管理系统里的预约记录
   - c. 告诉 Agent 本项目使用 Python 3.10，代码注释用中文
   - d. 教 Agent 按实验室的标准流程，把 ROS2 bag 文件转换成训练数据集
4. 在 `execute_with_hooks` 里，如果 Hook 脚本本身有 bug，每次都以退出码 1 退出，Agent 会怎样？这样的设计合理吗？
5. （需要已安装 Claude Code）把 `block_dangerous.py` 配置成 Claude Code 的 PreToolUse Hook，然后让它运行 `sudo ls`，观察会发生什么。

<details>
<summary>参考答案</summary>

**第 1 题**：模型只能看到「日志工具」这四个字，无法判断它处理的是哪种日志、什么时候该用，可能在该用的时候不用，也可能在无关的任务里误用。描述要写清「做什么」和「什么时候用」。

**第 2 题**：在 `RULES` 里加一行，例如：

```python
(r"\bgit\s+push\b.*(--force|\s-f\b)", "禁止强制推送"),
```

`\s-f\b` 要求 `-f` 前面是空白，避免误伤分支名里带 `-f` 的情况。测试时 `git push --force`、`git push -f origin main` 退出码应为 2，另外两条应为 0。正则写完一定要用正反两类例子都测一遍。

**第 3 题**：

- a. **Hook**（`PostToolUse`）：「每次都要做」，不能靠模型自觉
- b. **MCP**：需要访问外部系统，是一项新能力
- c. **`AGENTS.md`**：每次都需要知道的项目背景
- d. **Skill**：特定任务的流程，可以附带转换脚本，平时不需要加载

**第 4 题**：退出码 1 不是 2，所以不会阻止操作，只会打印「Hook 出错」的警告，工具照常执行。这样 Hook 出 bug 时 Agent 不会完全瘫痪，但安全检查也就悄悄失效了。所以 Hook 出错的警告不能忽视，写好的 Hook 要像第 2 题那样先单独测试。

**第 5 题**：Hook 以退出码 2 拦下命令，Claude Code 不会执行它，并把 stderr 里的原因告诉模型，模型通常会说明这个操作被阻止了。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出 Skill 的文件结构，以及 `name`、`description` 的作用 | Skill 是什么 |
| 解释渐进式加载的三个层级，以及它和上下文管理的关系 | 渐进式加载 |
| 说出写好 Skill 的几个要点 | 怎样写好一个 Skill |
| 解释 Hook 和提示词规则的区别 | Hook 是什么 |
| 说出 Hook 通过 stdin 接收事件、用退出码返回结果，以及退出码 0 和 2 的含义 | Hook 和程序怎样交流 |
| 说明为什么字符串匹配的 Hook 只是安全网 | Hook 不是万能的 |
| 针对一个需求，在 `AGENTS.md`、Skill、MCP、Hook 中选出合适的机制 | 怎样选 |

前半周的原理部分到这里就结束了。从下一节开始，我们转向实际使用：怎样和 Agent 高效地合作。

下一节：[06-写好提示词](./06-写好提示词.ipynb)